# TinySleepNet baseline on the same protocol (Kaggle run)

Trains only the TinySleepNet baseline (Supratak & Guo 2020, CNN + LSTM over 12 consecutive 5-s windows = 60 s) on the same 103 subjects, the same 5 subject-wise folds, the same targets, the same multi-task heads and the same loss as the other deep baselines in `pmrt_baselines.ipynb`. Nothing else is trained and the earlier baseline results are not touched.

**Kaggle:** Settings: Accelerator = GPU T4, Internet = On. `Save Version` -> **Save & Run All (Commit)**. No new fold is started after 10.5 h; if the run stops at the limit, add this notebook's output as an input and commit again (finished folds are skipped).

**Output:** `baselines/runs/tinysleepnet2020/fold0.csv ... fold4.csv` (test predictions) and the same files zipped as `tinysleepnet2020_folds.zip` in the output panel. Download that zip.

In [ ]:
# Environment: Google Colab (data on Google Drive) or Kaggle (data downloaded from the shared Drive zip)
import os, sys, glob, shutil, subprocess, zipfile, time
try:
    import pywt
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
from pathlib import Path
KAGGLE = os.path.exists("/kaggle/working")
FILE_ID = "11VPVORnZ951jWSle1wMWC0DAuLz5aFC-"
T_START, TIME_LIMIT_H = time.time(), 10.5
if KAGGLE:
    ROOT = Path("/kaggle/working")
    hits = sorted(glob.glob("/kaggle/input/**/x.npy", recursive=True))
    if not hits:
        tmp = Path("/tmp/capdata")
        if not list(tmp.rglob("x.npy")):
            try:
                import gdown
            except ImportError:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True); import gdown
            tmp.mkdir(parents=True, exist_ok=True)
            gdown.download(id=FILE_ID, output="/tmp/cap.zip", quiet=False)
            zipfile.ZipFile("/tmp/cap.zip").extractall(tmp); os.remove("/tmp/cap.zip")
        hits = sorted(str(p) for p in tmp.rglob("x.npy"))
    assert hits, "x.npy not found in the zip / Kaggle input"
    DATA = Path(hits[0]).parent
    folds = sorted(glob.glob(str(DATA / "folds_5.json")) + glob.glob("/tmp/capdata/**/folds_5.json", recursive=True)
                   + glob.glob("/kaggle/input/**/folds_5.json", recursive=True))
    assert folds, "folds_5.json not found"
    FOLDS_FILE = Path(folds[0])
    for prev in glob.glob("/kaggle/input/**/baselines", recursive=True):
        shutil.copytree(prev, ROOT / "baselines", dirs_exist_ok=True)
        print("previous Kaggle output found and copied:", prev)
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    ROOT = Path("/content/drive/MyDrive/sleep_project")
    DATA, FOLDS_FILE = ROOT / "cap_dataset_cache", ROOT / "folds_5.json"
print("platform:", "Kaggle" if KAGGLE else "Colab", "| data:", DATA, "| folds:", FOLDS_FILE, "| outputs:", ROOT)
print("files in data folder:", sorted(os.listdir(DATA)))

In [ ]:
# Shared helpers, identical to pmrt_mt3.py (labels, data loading, subject-wise folds, subject-balanced batches)
import math, types, numpy as np, pandas as pd, torch
DISORDERS = ["Healthy", "NFLE", "RBD", "PLM", "Insomnia", "Narcolepsy"]
STAGES = ["N1", "N2", "N3", "REM"]
IGNORE = -100

def load_data(cache_dir):
    X = np.load(Path(cache_dir) / "x.npy")
    idx = pd.read_csv(Path(cache_dir) / "index.csv")
    idx["dis"] = idx.group.map({g: i for i, g in enumerate(DISORDERS)}).fillna(IGNORE).astype(int)
    idx["stg"] = idx.stage.map({s: i for i, s in enumerate(STAGES)}).astype(int)
    return X, idx

def fold_rows(idx, fold, k_test, k=5):
    f = idx.record.map(fold).to_numpy()
    k_val = (k_test + 1) % k
    return {"train": np.where((f != k_test) & (f != k_val))[0],
            "val": np.where(f == k_val)[0], "test": np.where(f == k_test)[0]}

class SubjectBalancedBatches:
    def __init__(self, X, Y, rows, rec, batch, device, per_subject=1500, shuffle=True):
        self.X, self.Y, self.b, self.dev, self.k, self.sh = X, Y, batch, device, per_subject, shuffle
        self.groups = [rows[rec[rows] == r] for r in np.unique(rec[rows])] if shuffle else None
        self.rows = rows
    def _epoch_rows(self):
        if not self.sh: return self.rows
        pick = [g if len(g) <= self.k else np.random.choice(g, self.k, replace=False) for g in self.groups]
        return np.random.permutation(np.concatenate(pick))
    def __len__(self):
        n = sum(min(len(g), self.k) for g in self.groups) if self.sh else len(self.rows)
        return math.ceil(n / self.b)
    def __iter__(self):
        r = self._epoch_rows()
        for i in range(0, len(r), self.b):
            j = np.sort(r[i:i + self.b])
            x = torch.from_numpy(np.asarray(self.X[j], dtype=np.float32)).to(self.dev, non_blocking=True)
            y = {k: torch.from_numpy(v[j]).to(self.dev, non_blocking=True) for k, v in self.Y.items()}
            yield x, y, j

M = types.SimpleNamespace(DISORDERS=DISORDERS, STAGES=STAGES, IGNORE=IGNORE, load_data=load_data, fold_rows=fold_rows,
                          SubjectBalancedBatches=SubjectBalancedBatches)

def make_sequences(idx, rows, L=12, max_gap=60.0, offset=0):
    sub = idx.iloc[rows][["record", "t"]].assign(row=rows).sort_values(["record", "t"])
    seqs = []
    for _, g in sub.groupby("record", sort=False):
        r, t = g.row.to_numpy(), g.t.to_numpy()
        breaks = np.where(np.diff(t) > max_gap)[0] + 1
        for seg in np.split(r, breaks):
            o = offset % L if len(seg) > L else 0
            cuts = list(range(o, len(seg), L)) if o else list(range(0, len(seg), L))
            if o: cuts = [0] + cuts
            for a, b in zip(cuts, cuts[1:] + [len(seg)]):
                chunk = seg[a:b]
                if len(chunk): seqs.append(np.pad(chunk, (0, L - len(chunk)), constant_values=-1))
    return np.array(seqs, dtype=np.int64).reshape(-1, L)

class SequenceBatches:
    def __init__(self, X, Y, idx, rows, batch_seq, device, L=12, per_subject=1500, train=True):
        self.X, self.Y, self.idx, self.rows, self.b, self.dev, self.L = X, Y, idx, rows, batch_seq, device, L
        self.k, self.train = per_subject // L, train
        self.rec = idx.record.to_numpy()
        self.fixed = None if train else make_sequences(idx, rows, L)
    def _epoch(self):
        if not self.train: return self.fixed
        s = make_sequences(self.idx, self.rows, self.L, offset=np.random.randint(self.L))
        own = self.rec[s[:, 0]]
        keep = np.concatenate([np.random.permutation(np.where(own == r)[0])[:self.k] for r in np.unique(own)])
        return s[np.random.permutation(keep)]
    def __len__(self):
        n = len(self.fixed) if not self.train else len(np.unique(self.rec[self.rows])) * self.k
        return math.ceil(n / self.b)
    def __iter__(self):
        S = self._epoch()
        for i in range(0, len(S), self.b):
            s = S[i:i + self.b]; m = s >= 0
            j = s[m]
            x = torch.from_numpy(np.asarray(self.X[j], dtype=np.float32)).to(self.dev, non_blocking=True)
            y = {k: torch.from_numpy(v[j]).to(self.dev, non_blocking=True) for k, v in self.Y.items()}
            yield x, torch.from_numpy(m).to(self.dev), y, j

M.make_sequences, M.SequenceBatches = make_sequences, SequenceBatches

In [ ]:
# Settings, GPU, data and the same subject-wise folds as PMRT
import sys, json, time, importlib, warnings, numpy as np, pandas as pd, pywt, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, AdaBoostClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, cohen_kappa_score, f1_score
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")

OUT = ROOT / "baselines"
N_FOLDS, SEED, PER_SUBJECT = 5, 42, 1500
WAVELET, LEVEL, N_LEARNERS = "bior2.2", 5, 30
RUN_NAME, BATCH, EPOCHS, PATIENCE = "cheng2023_mmldms", 256, 30, 5
LR, MOMENTUM, WD, N_FFT, HOP = 0.01, 0.9, 5e-4, 128, 8
MODALITIES = {"eeg": 0, "emg": 1}
ONLY_NEW_METHODS = True

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "- WARNING: no GPU (Cheng et al. part will be very slow)")
TABB = ROOT / "tables" / "baselines"
for p in [OUT / "runs", TABB] + [ROOT / "figures" / e / "baselines" for e in ("png", "pdf", "svg")]:
    p.mkdir(parents=True, exist_ok=True)
X, idx = M.load_data(str(DATA))
FOLD = json.loads(FOLDS_FILE.read_text())
REC = idx.record.to_numpy()
DIS, STG = M.DISORDERS, M.STAGES
y_hr, y_lrm = idx.hr.to_numpy(np.float32), np.log(np.maximum(idx.rmssd.to_numpy(np.float32), 1.0))
y_dis, y_stg = idx.dis.to_numpy(), idx.stg.to_numpy()
LRM = y_lrm
RUN = OUT / "runs" / RUN_NAME; RUN.mkdir(parents=True, exist_ok=True)
print("windows:", X.shape, "| subjects:", idx.record.nunique())

In [ ]:
# Cheng et al. 2023 (MML-DMS): STFT spectrogram -> one VGG-style CNN per modality -> fusion NN with two hidden layers of 128
class Spectrogram(nn.Module):
    def __init__(self):
        super().__init__(); self.register_buffer("w", torch.hann_window(N_FFT))
    def forward(self, x):
        s = torch.log1p(torch.stft(x, N_FFT, HOP, window=self.w, return_complex=True).abs())
        s = (s - s.mean((1, 2), keepdim=True)) / (s.std((1, 2), keepdim=True) + 1e-6)
        return s[:, None]

def vgg_block(c_in, c_out, n):
    layers = []
    for _ in range(n):
        layers += [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]; c_in = c_out
    return nn.Sequential(*layers, nn.MaxPool2d(2))

HEADS = {"dis": len(M.DISORDERS), "stg": len(M.STAGES), "hr": 1, "lrm": 1}

class ModalityCNN(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.ch = ch; self.spec = Spectrogram()
        self.features = nn.Sequential(vgg_block(1, 32, 2), vgg_block(32, 64, 2), vgg_block(64, 128, 3), vgg_block(128, 256, 3),
                                      nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, 256), nn.ReLU(inplace=True), nn.Dropout(0.5))
        self.heads = nn.ModuleDict({k: nn.Linear(256, n) for k, n in HEADS.items()})
    def forward(self, x):
        h = self.features(self.spec(x[:, self.ch]))
        return {k: v(h) for k, v in self.heads.items()}

class FusionNN(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_in, 128), nn.ReLU(inplace=True), nn.Linear(128, 128), nn.ReLU(inplace=True))
        self.heads = nn.ModuleDict({k: nn.Linear(128, n) for k, n in HEADS.items()})
    def forward(self, z):
        h = self.body(z)
        return {k: v(h) for k, v in self.heads.items()}

def level1_vector(out):
    return torch.cat([F.softmax(out["dis"].float(), 1), F.softmax(out["stg"].float(), 1), out["hr"].float(), out["lrm"].float()], 1)

def loss_fn(out, y):
    l = F.cross_entropy(out["stg"].float(), y["stg"]) + F.mse_loss(out["hr"][:, 0].float(), y["hr"]) + F.mse_loss(out["lrm"][:, 0].float(), y["lrm"])
    if (y["dis"] != M.IGNORE).any(): l = l + F.cross_entropy(out["dis"].float(), y["dis"], ignore_index=M.IGNORE)
    return l

In [ ]:
# Deep EEG baselines - architectures adapted to 5-s windows at 125 Hz: EEGNet (2018), AttnSleep (2021), TinySleepNet (2020, sequence model)
class Heads(nn.Module):
    def __init__(self, d):
        super().__init__(); self.h = nn.ModuleDict({k: nn.Linear(d, n) for k, n in HEADS.items()})
    def forward(self, z): return {k: v(z) for k, v in self.h.items()}

class EEGNet(nn.Module):
    def __init__(self, C=2, F1=8, D=2, F2=16, k=64, p=0.25):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(1, F1, (1, k), padding=(0, k // 2), bias=False), nn.BatchNorm2d(F1),
                                nn.Conv2d(F1, F1 * D, (C, 1), groups=F1, bias=False), nn.BatchNorm2d(F1 * D), nn.ELU(),
                                nn.AvgPool2d((1, 4)), nn.Dropout(p))
        self.b2 = nn.Sequential(nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
                                nn.Conv2d(F1 * D, F2, 1, bias=False), nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 8)),
                                nn.Dropout(p), nn.Flatten())
        with torch.no_grad(): n = self.b2(self.b1(torch.zeros(2, 1, C, 625))).shape[1]
        self.head = Heads(n)
    def forward(self, x): return self.head(self.b2(self.b1(x[:, None])))

def conv_bn(ci, co, k, s=1, act=nn.ReLU):
    return nn.Sequential(nn.Conv1d(ci, co, k, s, k // 2, bias=False), nn.BatchNorm1d(co), act())

class SEResidual(nn.Module):
    def __init__(self, ci, co, r=4):
        super().__init__()
        self.body = nn.Sequential(nn.Conv1d(ci, co, 1, bias=False), nn.BatchNorm1d(co), nn.ReLU(inplace=True),
                                  nn.Conv1d(co, co, 1, bias=False), nn.BatchNorm1d(co))
        self.se = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(co, max(co // r, 4)), nn.ReLU(inplace=True),
                                nn.Linear(max(co // r, 4), co), nn.Sigmoid())
        self.down = nn.Sequential(nn.Conv1d(ci, co, 1, bias=False), nn.BatchNorm1d(co))
    def forward(self, x):
        h = self.body(x); return F.relu(h * self.se(h)[:, :, None] + self.down(x))

class AttnSleep(nn.Module):
    def __init__(self, afr=30, d_model=80, heads=5, layers=2, p=0.5):
        super().__init__()
        self.b1 = nn.Sequential(conv_bn(1, 64, 50, 6, nn.GELU), nn.MaxPool1d(8, 2, 4), nn.Dropout(p),
                                conv_bn(64, 128, 8, 1, nn.GELU), conv_bn(128, 128, 8, 1, nn.GELU), nn.MaxPool1d(4, 4, 2))
        self.b2 = nn.Sequential(conv_bn(1, 64, 200, 25, nn.GELU), nn.MaxPool1d(4, 2, 2), nn.Dropout(p),
                                conv_bn(64, 128, 7, 1, nn.GELU), conv_bn(128, 128, 7, 1, nn.GELU), nn.MaxPool1d(2, 2, 1))
        self.drop = nn.Dropout(p); self.afr = SEResidual(128, afr)
        with torch.no_grad(): t = self.b1(torch.zeros(2, 1, 625)).shape[-1] + self.b2(torch.zeros(2, 1, 625)).shape[-1]
        self.proj = nn.Linear(t, d_model)
        layer = nn.TransformerEncoderLayer(d_model, heads, 120, 0.1, batch_first=True)
        self.tce = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = Heads(afr * d_model)
    def forward(self, x):
        e = x[:, :1]; h = self.drop(torch.cat([self.b1(e), self.b2(e)], -1))
        h = self.tce(self.proj(self.afr(h)))
        return self.head(h.flatten(1))

class TinySleepNet(nn.Module):
    def __init__(self, C=2, h=128, p=0.5):
        super().__init__()
        self.cnn = nn.Sequential(nn.Conv1d(C, 128, 32, 4, 16, bias=False), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
                                 nn.MaxPool1d(4, 4, 2), nn.Dropout(p),
                                 conv_bn(128, 128, 8), conv_bn(128, 128, 8), conv_bn(128, 128, 8),
                                 nn.MaxPool1d(4, 4, 2), nn.Flatten(), nn.Dropout(p))
        with torch.no_grad(): n = self.cnn.eval()(torch.zeros(2, C, 625)).shape[1]
        self.cnn.train()
        self.lstm = nn.LSTM(n, h, batch_first=True)
        self.drop = nn.Dropout(p)
        self.head = Heads(h)
    def forward(self, x, mask):
        f = self.cnn(x)
        V = f.new_zeros(*mask.shape, f.shape[1]); V[mask] = f
        o, _ = self.lstm(V)
        return self.head(self.drop(o)[mask])

DEEP = {"eegnet2018": {"make": EEGNet, "seq": False,
                       "paper": "EEGNet (Lawhern et al. 2018, J Neural Eng) - EEG + chin EMG as 2 channels"},
        "attnsleep2021": {"make": AttnSleep, "seq": False,
                          "paper": "AttnSleep (Eldele et al. 2021, IEEE TNSRE) - EEG, multi-resolution CNN + AFR + attention"},
        "tinysleepnet2020": {"make": TinySleepNet, "seq": True,
                             "paper": "TinySleepNet (Supratak & Guo 2020, EMBC) - EEG + chin EMG as 2 channels, CNN + LSTM over 12 windows (60 s)"}}
for tag_, c_ in DEEP.items():
    m_ = c_["make"]().to(DEV).eval()
    with torch.no_grad():
        o_ = m_(torch.randn(24, 2, 625, device=DEV), torch.ones(2, 12, dtype=torch.bool, device=DEV)) if c_["seq"] else m_(torch.randn(4, 2, 625, device=DEV))
    print(f"{tag_:18s} {sum(p.numel() for p in m_.parameters()) / 1e6:.2f} M parameters | outputs {[tuple(v.shape) for v in o_.values()]}")
    del m_

In [ ]:
# TinySleepNet baseline - training (only this model): same folds, same multi-task heads and loss as Cheng 2023 (TinySleepNet sees 12 consecutive windows, like PMRT), Adam, early stopping on validation; resumable
DL_EPOCHS, DL_PATIENCE, DL_LR, DL_WD = 30, 5, 1e-3, 1e-3

def unpack(batch, seq):
    return (batch[0], batch[1], batch[2], batch[3]) if seq else (batch[0], None, batch[1], batch[2])

@torch.no_grad()
def predict_deep(model, loader, seq):
    model.eval(); outs, J, tot, nb = {}, [], 0.0, 0
    for batch in loader:
        x, m, y, j = unpack(batch, seq)
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            o = model(x, m) if seq else model(x)
        tot += loss_fn(o, y).item(); nb += 1
        for k, v in o.items(): outs.setdefault(k, []).append(v.float().cpu())
        J.append(j)
    J = np.concatenate(J); order = np.argsort(J, kind="stable")
    return J[order], {k: torch.cat(v)[order] for k, v in outs.items()}, tot / nb

def train_deep(cfg, tr, va, path):
    model = cfg["make"]().to(DEV); seq = cfg["seq"]
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEV)); return model
    opt = torch.optim.Adam(model.parameters(), lr=DL_LR, weight_decay=DL_WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, DL_EPOCHS)
    scaler = torch.amp.GradScaler(enabled=DEV == "cuda"); best, bad = float("inf"), 0
    for ep in range(1, DL_EPOCHS + 1):
        t0 = time.time(); model.train(); tot, nb = 0.0, 0
        for batch in tr:
            x, m, y, _ = unpack(batch, seq)
            with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
                loss = loss_fn(model(x, m) if seq else model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0); scaler.step(opt); scaler.update(); tot += loss.item(); nb += 1
        sched.step(); _, _, vl = predict_deep(model, va, seq)
        flag = ""
        if vl < best: best, bad, flag = vl, 0, " *best*"; torch.save(model.state_dict(), path.with_suffix(".tmp"))
        else: bad += 1
        print(f"    ep {ep:2d} | train {tot / nb:.3f} | val {vl:.3f} | {time.time() - t0:.0f}s{flag}", flush=True)
        if bad >= DL_PATIENCE: break
    path.with_suffix(".tmp").rename(path)
    model.load_state_dict(torch.load(path, map_location=DEV)); return model

for tag, cfg in ((t, c) for t, c in DEEP.items() if t == "tinysleepnet2020"):
    run = OUT / "runs" / tag; run.mkdir(parents=True, exist_ok=True)
    print("\n" + cfg["paper"])
    for k in range(N_FOLDS):
        f = run / f"fold{k}.csv"
        if f.exists():
            print(f"  fold {k}: done - skipped"); continue
        if time.time() - T_START > TIME_LIMIT_H * 3600:
            print("  time limit reached - run again to continue"); break
        torch.manual_seed(SEED + k); np.random.seed(SEED + k)
        rows = M.fold_rows(idx, FOLD, k, N_FOLDS)
        mu, sd = y_hr[rows["train"]].mean(), y_hr[rows["train"]].std()
        lm, ls = LRM[rows["train"]].mean(), LRM[rows["train"]].std()
        Y = {"hr": ((y_hr - mu) / sd).astype(np.float32), "lrm": ((LRM - lm) / ls).astype(np.float32),
             "dis": idx.dis.to_numpy(np.int64), "stg": idx.stg.to_numpy(np.int64)}
        if cfg["seq"]:
            tr = M.SequenceBatches(X, Y, idx, rows["train"], 24, DEV, 12, PER_SUBJECT, train=True)
            va = M.SequenceBatches(X, Y, idx, rows["val"], 64, DEV, 12, train=False)
            te = M.SequenceBatches(X, Y, idx, rows["test"], 64, DEV, 12, train=False)
        else:
            tr = M.SubjectBalancedBatches(X, Y, rows["train"], REC, BATCH, DEV, PER_SUBJECT, shuffle=True)
            va = M.SubjectBalancedBatches(X, Y, rows["val"], REC, 1024, DEV, shuffle=False)
            te = M.SubjectBalancedBatches(X, Y, rows["test"], REC, 1024, DEV, shuffle=False)
        print(f"  ===== fold {k} =====")
        model = train_deep(cfg, tr, va, run / f"fold{k}_model.pt")
        j, o, _ = predict_deep(model, te, cfg["seq"])
        assert (j == np.sort(rows["test"])).all()
        out = idx.iloc[j][["record", "group", "stage", "clipped", "dis", "stg", "hr", "rmssd"]].reset_index(drop=True)
        out["fold"] = k
        out["hr_pred"] = o["hr"][:, 0].numpy() * sd + mu
        out["rmssd_pred"] = np.exp(o["lrm"][:, 0].numpy() * ls + lm)
        pd_, ps = F.softmax(o["dis"], 1).numpy(), F.softmax(o["stg"], 1).numpy()
        for i, c in enumerate(DIS): out[f"p_{c}"] = pd_[:, i]
        for i, s in enumerate(STG): out[f"p_{s}"] = ps[:, i]
        out.to_csv(f, index=False); print(f"  fold {k} saved")
print("\nall deep baselines finished")

In [ ]:
# Zip the TinySleepNet test predictions for download
import zipfile
src = OUT / "runs" / "tinysleepnet2020"
with zipfile.ZipFile(ROOT / "tinysleepnet2020_folds.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k in range(N_FOLDS):
        f = src / f"fold{k}.csv"
        if f.exists(): z.write(f, f"tinysleepnet2020/fold{k}.csv")
print("zipped folds:", sorted(p.name for p in src.glob("fold*.csv")), "->", ROOT / "tinysleepnet2020_folds.zip")